# Pandas Series and DataFrames — labeled tables

> **Explain it like I am five:** A NumPy array is an egg tray. A Pandas DataFrame is a school register: rows have labels, columns have names, and different columns can store different kinds of information.

This rewrite preserves the original Series, DataFrame, indexing, editing, CSV, and summary examples while fixing their order so every cell runs correctly.

## Learning goals

- create and inspect Series and DataFrames;
- choose correctly among `[]`, `loc`, `iloc`, `at`, and `iat`;
- add, update, and remove rows or columns safely;
- filter, sort, summarize, and group real sales data;
- avoid chained assignment and accidental mutation.


In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)


Pandas version: 3.0.3


In [2]:
from pathlib import Path

def data_path(filename):
    """Find a course data file whether Jupyter starts in the repo or lesson folder."""
    current = Path.cwd().resolve()
    lesson_parts = ("Complete-Python-Bootcamp-main", "10-Data Analysis With Python")
    candidates = [current / filename, current.joinpath(*lesson_parts, filename)]
    for parent in current.parents:
        candidates.extend([parent / filename, parent.joinpath(*lesson_parts, filename)])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r}. Start Jupyter inside this repository.")


## 1. Series: one labeled column

A Series has values and an index. If we do not supply labels, Pandas uses `0, 1, 2, ...`.


In [3]:
## Series
## A Pandas Series is a one-dimensional array-like object that can hold any data type.

data=[1,2,3,4,5]
series=pd.Series(data)
print("Series \n",series)
print(type(series))


Series 
 0    1
1    2
2    3
3    4
4    5
dtype: int64
<class 'pandas.Series'>


In [4]:
## Create a Series from dictionary
data={'a':1,'b':2,'c':3}
series_dict=pd.Series(data)
print(series_dict)


a    1
b    2
c    3
dtype: int64


In [5]:
data=[10,20,30]
index=['a','b','c']
labeled_series = pd.Series(data,index=index)
print(labeled_series)
print("label b:", labeled_series.loc['b'])
print("position 1:", labeled_series.iloc[1])


a    10
b    20
c    30
dtype: int64
label b: 20
position 1: 20


## 2. DataFrame: many labeled columns

A dictionary of equal-length lists is a natural way to create a DataFrame. Dictionary keys become column names.


In [6]:
## Dataframe
## create a Dataframe from a dictionary of lists

data={
    'Name':['Krish','John','Jack'],
    'Age':[25,30,45],
    'City':['Bangalore','New York','Florida']
}
df=pd.DataFrame(data)
print(df)
print(type(df))


    Name  Age       City
0  Krish   25  Bangalore
1   John   30   New York
2   Jack   45    Florida
<class 'pandas.DataFrame'>


In [7]:
## Create a Data frame From a List of Dictionaries

data=[
    {'Name':'Krish','Age':32,'City':'Bangalore'},
    {'Name':'John','Age':34,'City':'Bangalore'},
    {'Name':'Bappy','Age':32,'City':'Bangalore'},
    {'Name':'JAck','Age':32,'City':'Bangalore'}
]
people_df=pd.DataFrame(data)
print(people_df)
print(type(people_df))


    Name  Age       City
0  Krish   32  Bangalore
1   John   34  Bangalore
2  Bappy   32  Bangalore
3   JAck   32  Bangalore
<class 'pandas.DataFrame'>


`people_df` is kept separate from the later sales table. The original notebook replaced `df` with a 240-row CSV and then tried to access `Name` and add three salaries, which caused errors. A clear variable name prevents that mix-up.


## 3. Inspect before you analyze

Use `head`, `tail`, `shape`, `columns`, `dtypes`, and `info()` to understand a table before selecting columns.


In [8]:
print(people_df.head(5))
print(people_df.tail(2))
print("shape:", people_df.shape)
print("columns:", people_df.columns.tolist())
print("dtypes:\n", people_df.dtypes)


    Name  Age       City
0  Krish   32  Bangalore
1   John   34  Bangalore
2  Bappy   32  Bangalore
3   JAck   32  Bangalore
    Name  Age       City
2  Bappy   32  Bangalore
3   JAck   32  Bangalore
shape: (4, 3)
columns: ['Name', 'Age', 'City']
dtypes:
 Name      str
Age     int64
City      str
dtype: object


## 4. Selecting data

| Tool | Chooses by | Typical use |
|---|---|---|
| `df['Name']` | column label | one Series |
| `df[['Name', 'Age']]` | column labels | DataFrame with chosen columns |
| `df.loc[row_label, column_label]` | labels | labeled selection |
| `df.iloc[row_position, column_position]` | positions | numbered selection |
| `df.at[label, label]` | labels | one fast scalar |
| `df.iat[position, position]` | positions | one fast scalar |


In [9]:
### Accessing Data From Dataframe
df = people_df.copy()
print(df)
print(df['Name'])


    Name  Age       City
0  Krish   32  Bangalore
1   John   34  Bangalore
2  Bappy   32  Bangalore
3   JAck   32  Bangalore
0    Krish
1     John
2    Bappy
3     JAck
Name: Name, dtype: str


In [10]:
print("loc row label 0:\n", df.loc[0])
print("iloc row position 0:\n", df.iloc[0])


loc row label 0:
 Name        Krish
Age            32
City    Bangalore
Name: 0, dtype: object
iloc row position 0:
 Name        Krish
Age            32
City    Bangalore
Name: 0, dtype: object


In [11]:
## Accessing a specified element
print(df.at[2,'Age'])
print(df.at[2,'Name'])


32
Bappy


In [12]:
## Accessing a specified element using iat
print(df.iat[2,2])


Bangalore


`loc` includes both ends of a label slice, while `iloc` excludes the stop position like normal Python slicing.


In [13]:
print("loc labels 0 through 2:\n", df.loc[0:2, ['Name', 'Age']])
print("iloc positions 0 through 1:\n", df.iloc[0:2, 0:2])


loc labels 0 through 2:
     Name  Age
0  Krish   32
1   John   34
2  Bappy   32
iloc positions 0 through 1:
     Name  Age
0  Krish   32
1   John   34


## 5. Add, update, and remove data

The original three-value salary example belongs with the three-row DataFrame created from the dictionary.


In [14]:
### Data Manipulation with Dataframe
df = pd.DataFrame({
    'Name':['Krish','John','Jack'],
    'Age':[25,30,45],
    'City':['Bangalore','New York','Florida']
})

## Adding a column
df['Salary']=[50000,60000,70000]
print(df)


    Name  Age       City  Salary
0  Krish   25  Bangalore   50000
1   John   30   New York   60000
2   Jack   45    Florida   70000


In [15]:
## Remove a column
df.drop('Salary',axis=1,inplace=True)
print(df)


    Name  Age       City
0  Krish   25  Bangalore
1   John   30   New York
2   Jack   45    Florida


In [16]:
## Add age to the column
df['Age']=df['Age']+1
print(df)


    Name  Age       City
0  Krish   26  Bangalore
1   John   31   New York
2   Jack   46    Florida


In [17]:
df.drop(0,inplace=True)
print(df)


   Name  Age      City
1  John   31  New York
2  Jack   46   Florida


`axis=1` means columns; `axis=0` means rows. `inplace=True` changes the object and returns `None`. Many analysts prefer assignment because it makes the change visible:

```python
df = df.drop(columns='Salary')
```


## 6. Read the local sales CSV reliably

`data_path` finds the file whether Jupyter starts in the lesson folder or repository root. This keeps the original `pd.read_csv('sales_data.csv')` idea without the fragile working-directory assumption.


In [18]:
sales_df=pd.read_csv(data_path('sales_data.csv'))
sales_df.head(5)


,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Region,Payment Method
0,10001,2024-01-01,Electronics,iPhone 14 Pro,2,999.99,1999.98,North America,Credit Card
1,10002,2024-01-02,Home Appliances,Dyson V11 Vacuum,1,499.99,499.99,Europe,PayPal
2,10003,2024-01-03,Clothing,Levi's 501 Jeans,3,69.99,209.97,Asia,Debit Card
3,10004,2024-01-04,Books,The Da Vinci Code,4,15.99,63.96,North America,Credit Card
4,10005,2024-01-05,Beauty Products,Neutrogena Skincare Set,1,89.99,89.99,Europe,PayPal


In [19]:
sales_df.tail(5)


,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Region,Payment Method
235,10236,2024-08-23,Home Appliances,Nespresso Vertuo Next Coffee and Espresso Maker,1,159.99,159.99,Europe,PayPal
236,10237,2024-08-24,Clothing,Nike Air Force 1 Sneakers,3,90.00,270.00,Asia,Debit Card
237,10238,2024-08-25,Books,The Handmaid's Tale by Margaret Atwood,3,10.99,32.97,North America,Credit Card
238,10239,2024-08-26,Beauty Products,Sunday Riley Luna Sleeping Night Oil,1,55.00,55.00,Europe,PayPal
239,10240,2024-08-27,Sports,Yeti Rambler 20 oz Tumbler,2,29.99,59.98,Asia,Credit Card


In [20]:
print("shape:", sales_df.shape)
print("columns:", sales_df.columns.tolist())
sales_df.info()


shape: (240, 9)
columns: ['Transaction ID', 'Date', 'Product Category', 'Product Name', 'Units Sold', 'Unit Price', 'Total Revenue', 'Region', 'Payment Method']
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    240 non-null    int64  
 1   Date              240 non-null    str    
 2   Product Category  240 non-null    str    
 3   Product Name      240 non-null    str    
 4   Units Sold        240 non-null    int64  
 5   Unit Price        240 non-null    float64
 6   Total Revenue     240 non-null    float64
 7   Region            240 non-null    str    
 8   Payment Method    240 non-null    str    
dtypes: float64(2), int64(2), str(5)
memory usage: 32.0 KB


## 7. Statistical summaries

`describe()` summarizes numeric columns by default. `include='all'` also describes text columns, although some cells are naturally blank because text has no mean.


In [21]:
# Display the data types of each column
print("Data types:\n", sales_df.dtypes)

# Describe the DataFrame
print("Statistical summary:\n", sales_df.describe())


Data types:
 Transaction ID        int64
Date                    str
Product Category        str
Product Name            str
Units Sold            int64
Unit Price          float64
Total Revenue       float64
Region                  str
Payment Method          str
dtype: object
Statistical summary:
        Transaction ID  Units Sold   Unit Price  Total Revenue
count       240.00000  240.000000   240.000000     240.000000
mean      10120.50000    2.158333   236.395583     335.699375
std          69.42622    1.322454   429.446695     485.804469
min       10001.00000    1.000000     6.500000       6.500000
25%       10060.75000    1.000000    29.500000      62.965000
50%       10120.50000    2.000000    89.990000     179.970000
75%       10180.25000    3.000000   249.990000     399.225000
max       10240.00000   10.000000  3899.990000    3899.990000


In [22]:
sales_df.describe(include='all')


,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Region,Payment Method
count,240.00000,240,240,240,240.000000,240.000000,240.000000,240,240
unique,NaN,240,6,232,NaN,NaN,NaN,3,3
top,NaN,2024-01-01,Electronics,Dune by Frank Herbert,NaN,NaN,NaN,North America,Credit Card
freq,NaN,1,40,2,NaN,NaN,NaN,80,120
mean,10120.50000,NaN,NaN,NaN,2.158333,236.395583,335.699375,NaN,NaN
std,69.42622,NaN,NaN,NaN,1.322454,429.446695,485.804469,NaN,NaN
min,10001.00000,NaN,NaN,NaN,1.000000,6.500000,6.500000,NaN,NaN
25%,10060.75000,NaN,NaN,NaN,1.000000,29.500000,62.965000,NaN,NaN
50%,10120.50000,NaN,NaN,NaN,2.000000,89.990000,179.970000,NaN,NaN
75%,10180.25000,NaN,NaN,NaN,3.000000,249.990000,399.225000,NaN,NaN


## 8. Filter rows with boolean conditions

Each condition creates a True/False mask. Use `&` for AND, `|` for OR, and parentheses around each comparison.


In [23]:
high_value_asia = sales_df.loc[
    (sales_df['Region'] == 'Asia') & (sales_df['Total Revenue'] >= 500),
    ['Date', 'Product Name', 'Units Sold', 'Total Revenue']
]
high_value_asia.head()


,Date,Product Name,Units Sold,Total Revenue
8,2024-01-09,Nike Air Force 1,6,539.94
11,2024-01-12,Babolat Pure Drive Tennis Racket,3,599.97
35,2024-02-05,Peloton Bike,1,1895.00
65,2024-03-06,Garmin Edge 530,2,599.98
95,2024-04-05,Garmin Fenix 6X Pro,1,999.99


In [24]:
selected_categories = sales_df[
    sales_df['Product Category'].isin(['Electronics', 'Clothing'])
]
selected_categories.head()


,Transaction ID,Date,Product Category,Product Name,Units Sold,Unit Price,Total Revenue,Region,Payment Method
0,10001,2024-01-01,Electronics,iPhone 14 Pro,2,999.99,1999.98,North America,Credit Card
2,10003,2024-01-03,Clothing,Levi's 501 Jeans,3,69.99,209.97,Asia,Debit Card
6,10007,2024-01-07,Electronics,MacBook Pro 16-inch,1,2499.99,2499.99,North America,Credit Card
8,10009,2024-01-09,Clothing,Nike Air Force 1,6,89.99,539.94,Asia,Debit Card
12,10013,2024-01-13,Electronics,Samsung Galaxy Tab S8,2,749.99,1499.98,North America,Credit Card


## 9. Sort and find top records

Sorting changes the order, not the values. `nlargest` is convenient when you need the top few numeric rows.


In [25]:
print(sales_df.sort_values('Total Revenue', ascending=False).head(5)[
    ['Product Name', 'Units Sold', 'Total Revenue']
])

print("\nTop three with nlargest:")
print(sales_df.nlargest(3, 'Total Revenue')[['Product Name', 'Total Revenue']])


                  Product Name  Units Sold  Total Revenue
102        Canon EOS R5 Camera           1        3899.99
85                  LG OLED TV           2        2599.98
6          MacBook Pro 16-inch           1        2499.99
216  Apple MacBook Pro 16-inch           1        2399.00
0                iPhone 14 Pro           2        1999.98

Top three with nlargest:


            Product Name  Total Revenue
102  Canon EOS R5 Camera        3899.99
85            LG OLED TV        2599.98
6    MacBook Pro 16-inch        2499.99


## 10. Create columns without changing the source table

`.assign()` returns a new DataFrame. Here we calculate revenue per unit and a simple size label.


In [26]:
enriched_sales = sales_df.assign(
    Revenue_Per_Unit=lambda table: table['Total Revenue'] / table['Units Sold'],
    Order_Size=lambda table: pd.cut(
        table['Total Revenue'],
        bins=[0, 250, 1000, float('inf')],
        labels=['Small', 'Medium', 'Large']
    )
)
enriched_sales[['Total Revenue', 'Revenue_Per_Unit', 'Order_Size']].head()


,Total Revenue,Revenue_Per_Unit,Order_Size
0,1999.98,999.99,Large
1,499.99,499.99,Medium
2,209.97,69.99,Small
3,63.96,15.99,Small
4,89.99,89.99,Small


## 11. Dates and categorical data

Text dates do not know calendar rules. Convert them with `pd.to_datetime`. Repeated category text can use the memory-efficient `category` dtype.


In [27]:
typed_sales = sales_df.copy()
typed_sales['Date'] = pd.to_datetime(typed_sales['Date'])
typed_sales['Region'] = typed_sales['Region'].astype('category')

print(typed_sales.dtypes)
print("date range:", typed_sales['Date'].min(), "to", typed_sales['Date'].max())


Transaction ID               int64
Date                datetime64[us]
Product Category               str
Product Name                   str
Units Sold                   int64
Unit Price                 float64
Total Revenue              float64
Region                    category
Payment Method                 str
dtype: object
date range: 2024-01-01 00:00:00 to 2024-08-27 00:00:00


## 12. Group and summarize

`groupby` is like sorting receipts into labeled baskets, then calculating one answer per basket.


In [28]:
category_summary = (
    sales_df.groupby('Product Category', as_index=False)
    .agg(
        Orders=('Transaction ID', 'count'),
        Units=('Units Sold', 'sum'),
        Revenue=('Total Revenue', 'sum')
    )
    .sort_values('Revenue', ascending=False)
)
category_summary


,Product Category,Orders,Units,Revenue
3,Electronics,40,66,34982.41
4,Home Appliances,40,59,18646.16
5,Sports,40,88,14326.52
2,Clothing,40,145,8128.93
0,Beauty Products,40,46,2621.90
1,Books,40,114,1861.93


## 13. Safe assignment and copies

A filtered result may or may not share memory with its source. Make intent explicit with `.copy()`, then assign through `.loc`.


In [29]:
electronics = sales_df.loc[sales_df['Product Category'] == 'Electronics'].copy()
electronics.loc[:, 'Discounted Revenue'] = electronics['Total Revenue'] * 0.90
electronics[['Product Name', 'Total Revenue', 'Discounted Revenue']].head()


,Product Name,Total Revenue,Discounted Revenue
0,iPhone 14 Pro,1999.98,1799.982
6,MacBook Pro 16-inch,2499.99,2249.991
12,Samsung Galaxy Tab S8,1499.98,1349.982
18,Garmin Forerunner 945,999.98,899.982
24,Bose QuietComfort 35 Headphones,299.99,269.991


Avoid chained assignment such as `df[df['x'] > 0]['y'] = 1`. It is ambiguous and may not update the original. Use `df.loc[df['x'] > 0, 'y'] = 1`.


## 14. Common mistakes

- **Assuming columns exist:** inspect `df.columns` first.
- **Confusing labels and positions:** use `loc` for labels and `iloc` for positions.
- **Adding a wrong-length list:** the list length must match the number of rows; a scalar broadcasts.
- **Forgetting parentheses in combined filters:** each comparison needs parentheses.
- **Using chained assignment:** use `.loc[...] = ...`.
- **Overusing `inplace=True`:** assignment is often easier to read and chain.
- **Treating date text as dates:** convert with `pd.to_datetime`.
- **Mutating a table needed later:** call `.copy()` or use a new variable.


## 15. Mini practice

1. Select orders from Europe with at least two units.
2. Show only product name, payment method, and revenue.
3. Find the average revenue by region.
4. Add a boolean column indicating revenue above 1,000.


In [30]:
european_multi_unit = sales_df.loc[
    (sales_df['Region'] == 'Europe') & (sales_df['Units Sold'] >= 2),
    ['Product Name', 'Payment Method', 'Total Revenue']
]
print(european_multi_unit.head())

print("\nAverage revenue by region:")
print(sales_df.groupby('Region')['Total Revenue'].mean().round(2))

practice_result = sales_df.assign(
    Above_1000=sales_df['Total Revenue'].gt(1000)
)
print(practice_result[['Total Revenue', 'Above_1000']].head())


                        Product Name Payment Method  Total Revenue
7               Blueair Classic 480i         PayPal        1199.98
31                   Instant Pot Duo         PayPal         269.97
34          L'Oreal Revitalift Serum         PayPal          79.98
37                        Roomba i7+         PayPal        1599.98
58  Anastasia Beverly Hills Brow Wiz         PayPal          46.00

Average revenue by region:
Region
Asia             280.69
Europe           265.85
North America    460.55
Name: Total Revenue, dtype: float64
   Total Revenue  Above_1000
0        1999.98        True
1         499.99       False
2         209.97       False
3          63.96       False
4          89.99       False


## Easy revision cheat sheet

| Goal | Pandas pattern |
|---|---|
| Create Series | `pd.Series(values, index=labels)` |
| Create table | `pd.DataFrame(dictionary)` |
| Read CSV | `pd.read_csv(path)` |
| First/last rows | `df.head()`, `df.tail()` |
| Inspect | `df.shape`, `df.columns`, `df.dtypes`, `df.info()` |
| One column | `df['column']` |
| Several columns | `df[['a', 'b']]` |
| Select by label | `df.loc[rows, columns]` |
| Select by position | `df.iloc[rows, columns]` |
| One value | `df.at[label, col]`, `df.iat[row, col]` |
| Filter | `df.loc[condition]` |
| Sort | `df.sort_values('column')` |
| Add column | `df['new'] = expression` |
| Remove column | `df.drop(columns='name')` |
| Convert date | `pd.to_datetime(df['date'])` |
| Group summary | `df.groupby('key').agg(...)` |
| Safe subset edit | `subset = df.loc[mask].copy()` |

**Memory trick:** A Series is one labeled column; a DataFrame is a collection of labeled columns sharing the same row index.
